# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alex-jb/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

**Two signal checks first (I check the signals my rule leans on before I trust the rule).**
The rule uses two signals, both behind real FlyRank flags: **staleness** (behind the refresh flags) and **visibility/volume** (behind quick-win). I bucket each against the decline label and give a one-word verdict. The code cell prints both tables with `n`.

- **Signal A — staleness → decline (behind refresh flags): verdict `MIXED`.** Decline rate rises with age from 0-30d (~0.51) to 91-180d (~0.61) — but the very-stale 180+ bucket *reverses* to ~0.47, **below** the 0.54 base rate. A naive "sort by the stalest pages" rule would target pages that decline *less* than average. **This check saved the rule: I use a `≥90d` window, not the naive `≥180d`.**
- **Signal B — visibility (impressions_90d) → decline (behind quick-win): verdict `CONFIRMED`.** Visible pages (≥100 impressions) decline at ~0.60-0.63; low-visibility pages (<100) at ~0.39. Visibility is a real, monotonic decline signal — so the rule requires `visible`.

**The rule, in plain words:** *a page is a refresh candidate if it is **visible** (≥100 impressions in 90d) and **stale-ish** (not updated in ≥90 days), and I rank candidates by their impressions (biggest audience at risk first).*

- **Score:** `visible × stale_ish × impressions_90d` (transparent, no fitted weights).
- **Reason code (one):** `stale_visible`.
- **Action label:** `refresh` (non-candidates → `monitor`).

No leakage: inputs are `impressions_90d` and `days_since_last_update` (trailing features). `trend_direction` / `trend_pct` are **never** used (they define the label).

In [1]:
import os, sys, json, pandas as pd, numpy as np
if "google.colab" in sys.modules and not os.path.isdir("data/raw"):
    if not os.path.isdir("flyrank-ml-internship"):
        os.system("git clone --depth 1 https://github.com/alex-jb/flyrank-ml-internship flyrank-ml-internship")
    os.chdir("flyrank-ml-internship")
while not os.path.isdir("data/raw") and os.getcwd() != "/":
    os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
base = df["is_declining_label"].mean()
print(f"base decline rate: {base:.3f}  (n={len(df):,})\n")

# --- Signal A: staleness -> decline (behind refresh flags) ---
df["stale_bkt"] = pd.cut(df["days_since_last_update"], [-1,30,90,180,10**9], labels=["0-30","31-90","91-180","180+"])
a = df.groupby("stale_bkt", observed=True)["is_declining_label"].agg(n="size", decline_rate="mean").round(3)
print("Signal A  staleness -> decline   VERDICT: MIXED (rises to 91-180, then 180+ reverses below base)")
print(a.to_string(), "\n")

# --- Signal B: visibility -> decline (behind quick-win) ---
df["imp_bkt"] = pd.cut(df["impressions_90d"], [-1,100,1000,5000,10**9], labels=["<100","100-1k","1k-5k","5k+"])
b = df.groupby("imp_bkt", observed=True)["is_declining_label"].agg(n="size", decline_rate="mean").round(3)
print("Signal B  visibility -> decline   VERDICT: CONFIRMED (visible pages decline more)")
print(b.to_string())

base decline rate: 0.542  (n=30,000)

Signal A  staleness -> decline   VERDICT: MIXED (rises to 91-180, then 180+ reverses below base)
               n  decline_rate
stale_bkt                     
0-30       20480         0.511
31-90        175         0.589
91-180      9171         0.611
180+         174         0.471 

Signal B  visibility -> decline   VERDICT: CONFIRMED (visible pages decline more)
            n  decline_rate
imp_bkt                    
<100     8006         0.389
100-1k   8485         0.603
1k-5k    7359         0.634
5k+      6150         0.547


## 2. Build the ranked queue (writes the CSV)

Code the transparent score, attach the reason code + action, rank everything, and write
`work/outputs/baseline_action_score.csv`. Then evaluate **precision@K next to the base rate**,
and save the receipts to `work/outputs/w04_baseline_metrics.json` (committed; the CSV is not).

In [2]:
visible  = (df["impressions_90d"] >= 100).astype(int)
stale_ish= (df["days_since_last_update"] >= 90).astype(int)          # >=90, NOT >=180 (Signal A was MIXED)
cand = (visible & stale_ish).astype(bool)

df["score"]       = visible * stale_ish * df["impressions_90d"]
df["reason_code"] = np.where(cand, "stale_visible", "not_candidate")
df["action"]      = np.where(cand, "refresh", "monitor")

queue = df.sort_values("score", ascending=False)[
    ["content_id","score","reason_code","action","impressions_90d","days_since_last_update","avg_position","ctr","is_declining_label"]]

os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)   # gitignored by design; regenerates each run

def p_at_k(s,l,k):
    o=np.argsort(-np.asarray(s)); return float(np.asarray(l)[o[:k]].mean())
y=df["is_declining_label"].values
metrics={"base_rate":round(float(base),3), "n":int(len(df)),
         "candidates_stale_visible":int(cand.sum()),
         "precision_at_20":round(p_at_k(df["score"],y,20),3),
         "precision_at_50":round(p_at_k(df["score"],y,50),3),
         "precision_at_100":round(p_at_k(df["score"],y,100),3),
         "rule":"visible(impr>=100) * stale_ish(dslu>=90) * impressions_90d",
         "reason_code":"stale_visible","action":"refresh"}
json.dump(metrics, open("work/outputs/w04_baseline_metrics.json","w"), indent=2)
print("wrote work/outputs/baseline_action_score.csv  (", len(queue), "rows )")
print("wrote work/outputs/w04_baseline_metrics.json")
print(json.dumps(metrics, indent=2))
print(f"\n>> HONEST: precision@50 {metrics['precision_at_50']} is BELOW base {metrics['base_rate']} "
      "— ranking by raw impressions surfaces stable mega-traffic pages. This is the beatable baseline.")

wrote work/outputs/baseline_action_score.csv  ( 30000 rows )
wrote work/outputs/w04_baseline_metrics.json
{
  "base_rate": 0.542,
  "n": 30000,
  "candidates_stale_visible": 8119,
  "precision_at_20": 0.45,
  "precision_at_50": 0.44,
  "precision_at_100": 0.38,
  "rule": "visible(impr>=100) * stale_ish(dslu>=90) * impressions_90d",
  "reason_code": "stale_visible",
  "action": "refresh"
}

>> HONEST: precision@50 0.44 is BELOW base 0.542 — ranking by raw impressions surfaces stable mega-traffic pages. This is the beatable baseline.


## 3. Top-10 review (skeptic's eye)

For each of the top 10: the action, why it's there (reason code + the number driving it), and
**what would make it wrong**. The `wrong-if` is reasoned from the rule's logic — not from the
label (I show the label only as a post-hoc outcome, not as justification).

In [3]:
top = df.sort_values("score", ascending=False).head(10).reset_index(drop=True)
def wrong_if(r):
    p, c = r["avg_position"], r["ctr"]
    if p >= 20:  return f"ranks DEEP (pos {p:.0f}) - high volume but not competitive; a refresh rarely lifts a page this far down. Wrong if the impressions are branded/navigational noise, not winnable demand."
    if c >= 0.50: return f"already converts well (CTR {c:.2f}%); a decline here is likely lost query VOLUME, not content quality - a refresh can't recover demand-side loss."
    if c <= 0.10 and p <= 10: return f"good position (pos {p:.1f}) but low CTR ({c:.2f}%) - this is a title/meta fix, so action 'refresh' is too heavy; 'review_ctr' would fit better."
    return "mid-profile candidate; wrong if the page is seasonal or consolidating rather than genuinely decaying."
for i,r in top.iterrows():
    print(f"#{i+1:2d} {r['content_id'][:18]} | action={r['action']:7s} | why: {r['reason_code']}, {int(r['impressions_90d']):,} impressions, {int(r['days_since_last_update'])}d stale")
    print(f"     wrong-if: {wrong_if(r)}   [outcome label: {'declining' if r['is_declining_label'] else 'not-declining'}]")

# 1 content_5fe46e0499 | action=refresh | why: stale_visible, 517,715 impressions, 104d stale
     wrong-if: mid-profile candidate; wrong if the page is seasonal or consolidating rather than genuinely decaying.   [outcome label: declining]
# 2 content_2dba2b1f95 | action=refresh | why: stale_visible, 443,434 impressions, 104d stale
     wrong-if: ranks DEEP (pos 28) - high volume but not competitive; a refresh rarely lifts a page this far down. Wrong if the impressions are branded/navigational noise, not winnable demand.   [outcome label: not-declining]
# 3 content_2c2606c5d1 | action=refresh | why: stale_visible, 347,399 impressions, 104d stale
     wrong-if: already converts well (CTR 0.53%); a decline here is likely lost query VOLUME, not content quality - a refresh can't recover demand-side loss.   [outcome label: declining]
# 4 content_cb112fce36 | action=refresh | why: stale_visible, 309,910 impressions, 104d stale
     wrong-if: mid-profile candidate; wrong if the page is season

## 4. Weak picks + leakage check

**The whole rule is honestly weak — and I can say exactly why.** precision@50 (~0.44) sits *below*
the base rate (~0.54): ranking candidates by **raw impressions** pulls the biggest-traffic pages
to the top, and mega-traffic pages turn out to be *more stable*, not more declining. Several top
picks are deep-position, high-volume pages (likely branded/navigational) where a refresh wouldn't
help. The fix belongs to the Week-5 model: weight the signals (position, CTR-gap, staleness window)
instead of sorting by volume. **This baseline is the beatable bar.**

**Leakage check:** the score uses only `impressions_90d` and `days_since_last_update` — trailing
features, not future-window and not label-derived. `trend_direction` and `trend_pct` (which define
`is_declining_label`) are **never** inputs. Confirmed in the code cell below.

In [4]:
# Leakage guard: the label's source columns must not be in the score inputs.
score_inputs = ["impressions_90d", "days_since_last_update"]
label_sources = ["trend_direction", "trend_pct"]
leak = [c for c in score_inputs if c in label_sources]
print("score inputs:", score_inputs)
print("label-derived columns (forbidden as inputs):", label_sources)
print("leakage found:", leak if leak else "NONE  -> clean")
print(f"\nbaseline precision@50 = {p_at_k(df['score'],y,50):.3f}  vs  base rate {base:.3f}  "
      "-> weak on purpose, honestly reported, ready for the model to beat.")

score inputs: ['impressions_90d', 'days_since_last_update']
label-derived columns (forbidden as inputs): ['trend_direction', 'trend_pct']
leakage found: NONE  -> clean

baseline precision@50 = 0.440  vs  base rate 0.542  -> weak on purpose, honestly reported, ready for the model to beat.


## Self-check

- [x] Two signal verdicts with visible bucket tables and `n` (Signal A staleness = MIXED, Signal B visibility = CONFIRMED; both flag-linked)
- [x] One rule with a score, one reason code (`stale_visible`), and an action label (`refresh`)
- [x] Ranked queue written from the notebook to `work/outputs/baseline_action_score.csv`
- [x] Ten reviewed rows, each with a "what would make it wrong"
- [x] No future-window or label-derived inputs (leakage check passes)
- [x] Metrics receipt saved to `work/outputs/w04_baseline_metrics.json` (committed; CSV stays out of git)